# TrashScan — Path B3 (5 classes)

Este notebook assume que a estrutura **já está baixada/criada no volume** do RunPod:

- `/workspace/.venv`
- `/workspace/TrashScan`
- `/workspace/TACO`
- `/workspace/external_datasets`
- `/workspace/processed_5cls`
- `/workspace/runs`

O foco aqui é rodar o fluxo do **Path B** usando os scripts `.py` do projeto, sem refazer downloads e sem refazer merge/preprocessamento por padrão.

## 1) Imports, paths e utilitários

In [ ]:
from pathlib import Path
import os
import sys
import shlex
import subprocess
import shutil
import torch
import pandas as pd
import json

# Caminhos principais no RunPod
WORKSPACE = Path('/workspace')
REPO_ROOT = WORKSPACE / 'TrashScan'

DATA_DIR = REPO_ROOT / 'data'
TRAIN_DIR = REPO_ROOT / 'train' / 'paths'
EVAL_DIR = REPO_ROOT / 'eval'

EXTERNAL_DIR = WORKSPACE / 'external_datasets'
TACO_DIR = WORKSPACE / 'TACO'
PROCESSED_DIR = WORKSPACE / 'processed_5cls'

DATASET_YAML_PATH_A = PROCESSED_DIR / 'dataset_path_A.yaml'
DATASET_YAML_PATH_B = PROCESSED_DIR / 'dataset_path_B.yaml'

# Path A: necessário porque o Path B usa o melhor detector treinado no Path A
RUNS_PATH_A_DIR = WORKSPACE / 'runs' / 'path_A'

# Path B3: treino do classificador ViT-B/16 ImageNet21k
RUNS_PATH_B_DIR = WORKSPACE / 'runs' / 'path_B'
MLFLOW_DIR = Path('/root/mlflow')

# Resultados finais persistentes no volume
RESULTS_PATH_B_DIR = WORKSPACE / 'results_path_B'

# Scripts principais
TRAIN_PATH_B_SCRIPT = TRAIN_DIR / 'train_path_B.py'

for p in [RUNS_PATH_B_DIR, MLFLOW_DIR, RESULTS_PATH_B_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python             =', sys.executable)
print('REPO_ROOT          =', REPO_ROOT)
print('TACO_DIR           =', TACO_DIR)
print('EXTERNAL_DIR       =', EXTERNAL_DIR)
print('PROCESSED_DIR      =', PROCESSED_DIR)
print('DATASET_YAML_PATH_A=', DATASET_YAML_PATH_A)
print('DATASET_YAML_PATH_B=', DATASET_YAML_PATH_B)
print('RUNS_PATH_A_DIR    =', RUNS_PATH_A_DIR)
print('RUNS_PATH_B_DIR    =', RUNS_PATH_B_DIR)
print('MLFLOW_DIR         =', MLFLOW_DIR)
print('RESULTS_PATH_B_DIR =', RESULTS_PATH_B_DIR)
print('TRAIN_PATH_B_SCRIPT=', TRAIN_PATH_B_SCRIPT)


def run_cmd(cmd, cwd=WORKSPACE, env=None):
    """Roda comandos de forma previsível no notebook."""
    if isinstance(cmd, str):
        print('$', cmd)
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)

    print('$', ' '.join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)

## 2) Verificação dos arquivos principais

In [ ]:
required_paths = [
    TRAIN_PATH_B_SCRIPT,
    PROCESSED_DIR,
    RUNS_PATH_A_DIR,
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    print("Arquivos/pastas ausentes:")
    for p in missing:
        print(" -", p)
else:
    print("Tudo certo.")

## 3) Configuração de GPU e treino

In [ ]:
if torch.cuda.is_available():
    DEVICE = "0"
    gpu_name = torch.cuda.get_device_properties(0).name
else:
    DEVICE = "cpu"
    gpu_name = "cpu"

print("Device:", DEVICE)
print("GPU:", gpu_name)

EPOCHS = 100
BATCH = 8
LR = 5e-5
PATIENCE = 10
DET_CONF = 0.25

CLASSIFIERS = [
    "vit_b16_imagenet",
]

## 4) Encontrar o melhor peso do Path A

In [ ]:
PATH_A_RUN_DIRS = [
    WORKSPACE / "runs" / "path_A",
    WORKSPACE / "runs" / "path_A_5cls",
    WORKSPACE / "runs" / "path_A_refined_head",
]

def read_yolo_results(run_dir: Path):
    """
    Lê métricas de uma pasta de treino YOLO.
    Espera estrutura:
      run_dir/
        weights/best.pt
        results.csv
        args.yaml
    """
    best_pt = run_dir / "weights" / "best.pt"
    results_csv = run_dir / "results.csv"
    metrics_json = run_dir / "metrics.json"

    if not best_pt.exists():
        return None

    row = {
        "group": run_dir.parent.name,
        "model": run_dir.name,
        "run_dir": run_dir,
        "best_pt": best_pt,
        "mAP50_95": None,
        "mAP50": None,
        "precision": None,
        "recall": None,
        "source": None,
    }

    if results_csv.exists():
        df = pd.read_csv(results_csv)
        df.columns = [c.strip() for c in df.columns]

        # Melhor época por mAP50-95, se existir
        map95_col = "metrics/mAP50-95(B)"
        map50_col = "metrics/mAP50(B)"
        precision_col = "metrics/precision(B)"
        recall_col = "metrics/recall(B)"

        if map95_col in df.columns:
            best_idx = df[map95_col].idxmax()
        elif map50_col in df.columns:
            best_idx = df[map50_col].idxmax()
        else:
            best_idx = df.index[-1]

        best = df.loc[best_idx]

        row["mAP50_95"] = float(best[map95_col]) if map95_col in df.columns else None
        row["mAP50"] = float(best[map50_col]) if map50_col in df.columns else None
        row["precision"] = float(best[precision_col]) if precision_col in df.columns else None
        row["recall"] = float(best[recall_col]) if recall_col in df.columns else None
        row["source"] = "results.csv"
        return row

    # Fallback para metrics.json, se existir
    if metrics_json.exists():
        with open(metrics_json, "r") as f:
            m = json.load(f)

        row["mAP50_95"] = m.get("mAP50_95")
        row["mAP50"] = m.get("mAP50")
        row["precision"] = m.get("precision")
        row["recall"] = m.get("recall")
        row["source"] = "metrics.json"
        return row

    # Tem best.pt, mas sem métrica
    row["source"] = "weights_only"
    return row


records = []

for base_dir in PATH_A_RUN_DIRS:
    if not base_dir.exists():
        print(f"[warn] Pasta não encontrada: {base_dir}")
        continue

    for run_dir in sorted(base_dir.iterdir()):
        if not run_dir.is_dir():
            continue

        rec = read_yolo_results(run_dir)
        if rec is not None:
            records.append(rec)

df_detectors = pd.DataFrame(records)

if df_detectors.empty:
    raise FileNotFoundError(
        "Nenhum detector com weights/best.pt foi encontrado em: "
        + ", ".join(str(p) for p in PATH_A_RUN_DIRS)
    )

# Ordena pelo melhor critério disponível
df_ranked = df_detectors.copy()
df_ranked["rank_score"] = df_ranked["mAP50_95"].fillna(df_ranked["mAP50"]).fillna(-1)

df_ranked = df_ranked.sort_values(
    by=["rank_score", "mAP50", "precision", "recall"],
    ascending=False,
    na_position="last",
).reset_index(drop=True)

display_cols = [
    "group", "model", "mAP50_95", "mAP50", "precision", "recall", "source", "best_pt"
]

display(df_ranked[display_cols])

best_detector = df_ranked.iloc[0]
DETECTOR_WEIGHTS = Path(best_detector["best_pt"])

print("Melhor detector encontrado:")
print("Grupo :", best_detector["group"])
print("Modelo:", best_detector["model"])
print("mAP50-95:", best_detector["mAP50_95"])
print("mAP50:", best_detector["mAP50"])
print("Pesos:", DETECTOR_WEIGHTS)

if not DETECTOR_WEIGHTS.exists():
    raise FileNotFoundError(f"Detector não encontrado: {DETECTOR_WEIGHTS}")

## 5) Conferir estrutura do Path B

In [ ]:
for split in ["train", "val", "test"]:
    path_b_dir = PROCESSED_DIR / split / "path_B"
    print(split, path_b_dir, "->", path_b_dir.exists())

    for sub in ["images", "labels", "crops"]:
        p = path_b_dir / sub
        print("  ", sub, "->", p.exists())

## 6) Treino Path B3 — ViT-B/16 fine-tuning

In [ ]:
print("\nComando que será executado:")

print("\nParâmetros:")
print(f"Python executable: {sys.executable}")
print(f"Script: {TRAIN_PATH_B_SCRIPT}")
print(f"Detector weights: {DETECTOR_WEIGHTS}")
print(f"Crops dir: {PROCESSED_DIR}")
print(f"Output dir: {RUNS_PATH_B_DIR}")
print(f"Classifiers: {CLASSIFIERS}")
print(f"Epochs: {EPOCHS}")
print(f"Batch: {BATCH}")
print(f"Learning rate: {LR}")
print(f"Patience: {PATIENCE}")
print(f"Device: {DEVICE}")
print(f"Use YOLO crops: True")
print(f"Detection confidence: {DET_CONF}")
print(f"Crop cache dir: {RUNS_PATH_B_DIR / 'crop_cache'}")

In [ ]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--crops_dir", str(PROCESSED_DIR),
    "--output", str(RUNS_PATH_B_DIR),
    "--classifiers", *CLASSIFIERS,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--lr", str(LR),
    "--patience", str(PATIENCE),
    "--device", str(DEVICE),
    "--use_yolo_crops",
    "--det_conf", str(DET_CONF),
    "--crop_cache_dir", str(RUNS_PATH_B_DIR / "crop_cache"),
])

## Com TTA

In [ ]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--crops_dir", str(PROCESSED_DIR),
    "--output", str(RUNS_PATH_B_DIR),
    "--classifiers", *CLASSIFIERS,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--lr", str(LR),
    "--patience", str(PATIENCE),
    "--device", str(DEVICE),
    "--use_yolo_crops",
    "--det_conf", str(DET_CONF),
    "--tta",
    "--crop_cache_dir", str(RUNS_PATH_B_DIR / "crop_cache_tta"),
])

In [ ]:
run_dir = Path("/workspace/runs/path_B/vit_b16_imagenet")

print("best.pt:", (run_dir / "weights" / "best.pt").exists())
print("history.csv:", (run_dir / "history.csv").exists())
print("metrics.json:", (run_dir / "metrics.json").exists())

if (run_dir / "metrics.json").exists():
    print(json.loads((run_dir / "metrics.json").read_text()))

if (run_dir / "history.csv").exists():
    hist = pd.read_csv(run_dir / "history.csv")
    display(hist.tail())

## 7) Resumo dos treinos

In [ ]:
def run_cmd(cmd, cwd="/workspace", env=None, shell=False):
    print("$", " ".join(shlex.quote(str(x)) for x in cmd))
    if shell:
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)
   
TRAIN_PATH_B_SCRIPT = Path("/workspace/TrashScan/train/paths/train_path_B.py")
RUNS_PATH_B_DIR = Path("/workspace/runs/path_B")
DETECTOR_WEIGHTS = Path("/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt")
PROCESSED_DIR = Path("/workspace/processed_5cls")

In [ ]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--crops_dir", str(PROCESSED_DIR),
    "--output", str(RUNS_PATH_B_DIR),
    "--summarize",
])

## 8) Ver métricas do Path B3

In [ ]:
RUN_DIR = Path("/workspace/runs/path_B/vit_b16_imagenet")

history_path = RUN_DIR / "history.csv"
metrics_path = RUN_DIR / "metrics.json"
best_weights_path = RUN_DIR / "weights" / "best.pt"
cm_path = RUN_DIR / "confusion_matrix_vit_b16_imagenet.png"

print("Run dir:", RUN_DIR)
print("best.pt existe?", best_weights_path.exists())
print("history.csv existe?", history_path.exists())
print("metrics.json existe?", metrics_path.exists())
print("confusion matrix existe?", cm_path.exists())

if not history_path.exists():
    raise FileNotFoundError(f"history.csv não encontrado: {history_path}")

hist = pd.read_csv(history_path)

best_val_acc_idx = hist["val_acc"].idxmax()
best_val_f1_idx = hist["val_f1"].idxmax()

best_val_acc_row = hist.loc[best_val_acc_idx]
best_val_f1_row = hist.loc[best_val_f1_idx]

summary = {
    "run_dir": str(RUN_DIR),
    "best_weights": str(best_weights_path),
    "best_weights_exists": best_weights_path.exists(),

    "epochs_trained": int(hist["epoch"].max()),

    "best_epoch_by_val_acc": int(best_val_acc_row["epoch"]),
    "best_val_acc": float(best_val_acc_row["val_acc"]),
    "best_val_acc_val_f1": float(best_val_acc_row["val_f1"]),
    "best_val_acc_train_acc": float(best_val_acc_row["train_acc"]),
    "best_val_acc_train_loss": float(best_val_acc_row["train_loss"]),
    "best_val_acc_val_loss": float(best_val_acc_row["val_loss"]),

    "best_epoch_by_val_f1": int(best_val_f1_row["epoch"]),
    "best_val_f1": float(best_val_f1_row["val_f1"]),
    "best_val_f1_val_acc": float(best_val_f1_row["val_acc"]),
}

if metrics_path.exists():
    with open(metrics_path, "r") as f:
        test_metrics = json.load(f)

    summary.update({
        "test_accuracy": test_metrics.get("accuracy"),
        "test_precision_macro": test_metrics.get("precision"),
        "test_recall_macro": test_metrics.get("recall"),
        "test_f1_macro": test_metrics.get("f1"),
        "latency_ms": test_metrics.get("latency_ms"),
        "AP_plastic": test_metrics.get("AP_plastic"),
        "AP_paper": test_metrics.get("AP_paper"),
        "AP_metal": test_metrics.get("AP_metal"),
        "AP_glass": test_metrics.get("AP_glass"),
        "AP_other": test_metrics.get("AP_other"),
    })
else:
    print("\n⚠️ metrics.json não foi encontrado. Talvez o erro no MLflow tenha ocorrido antes de salvar as métricas.")

summary_df = pd.DataFrame([summary]).T.rename(columns={0: "value"})
display(summary_df)

if metrics_path.exists():
    print("\nMétricas de teste:")
    display(pd.DataFrame([test_metrics]))

In [ ]:
EVAL_PATH_B_COMBINED_SCRIPT = Path("/workspace/TrashScan/eval/evaluate_path_B_combined.py")

DETECTOR_WEIGHTS = Path("/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt")
CLASSIFIER_DIR = Path("/workspace/runs/path_B")
DATA_YAML = Path("/workspace/processed_5cls/dataset_path_B.yaml")
OUTPUT_DIR = Path("/workspace/results_path_B/b3_5cls")

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--classifier_dir", str(CLASSIFIER_DIR),
    "--classifiers", "vit_b16_imagenet",
    "--data_yaml", str(DATA_YAML),
    "--output", str(OUTPUT_DIR),
    "--device", "0",
    "--imgsz", "640",
    "--det_conf", "0.001",
    "--det_iou", "0.6",
]

print("$", " ".join(shlex.quote(str(x)) for x in cmd))

subprocess.run(
    [str(x) for x in cmd],
    cwd="/workspace",
    check=True,
)